<a href="https://colab.research.google.com/github/natalia-bioinfo/cerrado-sdm-barbatimao/blob/main/01_baixar_dados_gbif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Projeto Cerrado SDM: barbatimão")


Projeto Cerrado SDM: barbatimão


In [2]:
!pip install pygbif -q
import pandas as pd
from pygbif import species, occurrences

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 4.1 MB/s eta 0:00:00


In [5]:
info = species.name_backbone(scientificName="Stryphnodendron adstringens", taxonRank="SPECIES")
info

{'usage': {'key': '537LL',
  'name': 'Stryphnodendron adstringens (Mart.) Coville',
  'canonicalName': 'Stryphnodendron adstringens',
  'authorship': '(Mart.) Coville',
  'rank': 'SPECIES',
  'code': 'BOTANICAL',
  'status': 'ACCEPTED',
  'genericName': 'Stryphnodendron',
  'specificEpithet': 'adstringens',
  'type': 'SCIENTIFIC',
  'formattedName': '<i>Stryphnodendron</i> <i>adstringens</i> (Mart.) Coville'},
 'classification': [{'key': 'CS5HF', 'name': 'Eukaryota', 'rank': 'DOMAIN'},
  {'key': 'P', 'name': 'Plantae', 'rank': 'KINGDOM'},
  {'key': 'CMQ8S', 'name': 'Pteridobiotina', 'rank': 'SUBKINGDOM'},
  {'key': 'TP', 'name': 'Tracheophyta', 'rank': 'PHYLUM'},
  {'key': 'MG', 'name': 'Magnoliopsida', 'rank': 'CLASS'},
  {'key': '383', 'name': 'Fabales', 'rank': 'ORDER'},
  {'key': '623QT', 'name': 'Fabaceae', 'rank': 'FAMILY'},
  {'key': '9CM58', 'name': 'Stryphnodendron', 'rank': 'GENUS'},
  {'key': '537LL', 'name': 'Stryphnodendron adstringens', 'rank': 'SPECIES'}],
 'diagnostics'

In [6]:
chave = info["usage"]["key"]
print(chave, info["usage"]["canonicalName"])

537LL Stryphnodendron adstringens


In [7]:
teste = occurrences.search(taxonKey=chave, country="BR", hasCoordinate=True, limit=5)
print(teste["count"], "registros no total")
print(len(teste["results"]), "baixados neste teste")

1394 registros no total
5 baixados neste teste


In [8]:
registros = []
offset = 0

while True:
    resp = occurrences.search(
        taxonKey=chave,
        country="BR",
        hasCoordinate=True,
        hasGeospatialIssue=False,
        limit=300,
        offset=offset,
    )
    registros.extend(resp["results"])
    if resp["endOfRecords"]:
        break
    offset += 300

print(len(registros), "registros baixados")

1267 registros baixados


In [9]:
df = pd.DataFrame(registros)
colunas = ["key", "scientificName", "decimalLatitude", "decimalLongitude",
           "year", "stateProvince", "basisOfRecord", "datasetName"]
df = df[[c for c in colunas if c in df.columns]]
print(df.shape)
df.head()

(1267, 8)


,key,scientificName,decimalLatitude,decimalLongitude,year,stateProvince,basisOfRecord,datasetName
0,6130925469,Stryphnodendron adstringens (Mart.) Coville,-14.172442,-47.826203,2026.0,Goiás,HUMAN_OBSERVATION,iNaturalist research-grade observations
1,6131458424,Stryphnodendron adstringens (Mart.) Coville,-21.000422,-45.678128,2026.0,Minas Gerais,HUMAN_OBSERVATION,iNaturalist research-grade observations
2,6147183220,Stryphnodendron adstringens (Mart.) Coville,-18.198796,-43.605968,2026.0,Minas Gerais,HUMAN_OBSERVATION,iNaturalist research-grade observations
3,6479218041,Stryphnodendron adstringens (Mart.) Coville,-18.211985,-43.612570,2026.0,Minas Gerais,HUMAN_OBSERVATION,iNaturalist research-grade observations
4,6171117561,Stryphnodendron adstringens (Mart.) Coville,-15.940763,-47.829021,2026.0,Distrito Federal,HUMAN_OBSERVATION,iNaturalist research-grade observations


In [10]:
len(df)

1267

In [11]:
df["stateProvince"].value_counts()

,count
stateProvince,
Distrito Federal,266
Minas Gerais,238
São Paulo,205
Goiás,108
Bahia,86
Paraná,82
Mato Grosso do Sul,38
Maranhão,34
Sp,29


In [12]:
df["year"].isna().sum()

np.int64(42)

In [15]:
df["stateProvince"].value_counts(dropna=False)

,count
stateProvince,
Distrito Federal,266
Minas Gerais,238
São Paulo,205
Goiás,108
Bahia,86
Paraná,82
NaN,67
Mato Grosso do Sul,38
Maranhão,34


In [16]:
df["estado"] = df["stateProvince"].str.strip().str.lower()
df["estado"].value_counts(dropna=False)

,count
estado,
distrito federal,266
minas gerais,238
são paulo,205
goiás,108
bahia,86
paraná,82
NaN,67
mato grosso do sul,38
maranhão,34


In [18]:
correcao = {
    "mg": "minas gerais",
    "go": "goiás",
    # continue você, com as variações que apareceram na sua lista
}
df["estado"] = df["estado"].replace(correcao)

In [20]:
df["estado"].value_counts(dropna=False)

,count
estado,
distrito federal,266
minas gerais,251
são paulo,205
goiás,118
bahia,86
paraná,82
NaN,67
mato grosso do sul,38
maranhão,34


In [22]:
sorted(df["estado"].dropna().unique())

['alagoas',
 'ba',
 'bahia',
 'ceará',
 'df',
 'distrito federal',
 'goiás',
 'ma',
 'maranhão',
 'mato grosso',
 'mato grosso do sul',
 'minas gerais',
 'mt',
 'paraná',
 'piauí',
 'pr',
 'rio de janeiro',
 'rondônia',
 'sergipe',
 'sp',
 'são paulo',
 'tocantins']

In [24]:
df["estado"] = (
    df["estado"]
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("ascii")
)

In [25]:
sorted(df["estado"].dropna().unique())

['alagoas',
 'ba',
 'bahia',
 'ceara',
 'df',
 'distrito federal',
 'goias',
 'ma',
 'maranhao',
 'mato grosso',
 'mato grosso do sul',
 'minas gerais',
 'mt',
 'parana',
 'piaui',
 'pr',
 'rio de janeiro',
 'rondonia',
 'sao paulo',
 'sergipe',
 'sp',
 'tocantins']

In [26]:
correcao = {
    "ba": "bahia",
    "df": "distrito federal",
    "ma": "maranhao",
    "mt": "mato grosso",
    "pr": "parana",
    "sp": "sao paulo",
}
df["estado"] = df["estado"].replace(correcao)

In [27]:
df["estado"].value_counts(dropna=False)

,count
estado,
distrito federal,286
minas gerais,251
sao paulo,234
goias,118
bahia,89
parana,86
NaN,67
mato grosso do sul,38
maranhao,35
